[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/06_failure_modes.ipynb)


# Agentic Systems Foundations
## Notebook 06: How Agents Break, and How to See It
**Duration:** 20 min &nbsp;|&nbsp; **Mode:** Guided Analysis

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** the whole loop — under a microscope. Every step, examined for where it went wrong.


In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Makes `agent_core` importable whether you are in Colab, in a local venv,
# or running from a clone. Installs nothing you do not need: the package's
# only hard dependency is the Python standard library.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    # openai for the real provider; jsonschema + langchain for the parallel
    # mappings shown in notebook 03. All optional — the notebook degrades
    # gracefully if any is missing.
    _pip("openai", "python-dotenv", "jsonschema", "langchain-core")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

# Where the Acme data lives — the tools resolve this automatically, but we
# print it so a path problem is visible immediately rather than as an empty
# search result three cells later.
from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDER  (works with NO key at all)
# ============================================================
# Default stack = OpenAI gpt-4o-mini with native tool calling.
# In Colab the key is read from the SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON -> re-run this cell.
#
# With NO key we fall back to MockToolCallLLM. Read that name literally:
# unlike a text-only mock, it DECIDES TOOL CALLS, so the entire agent loop —
# every notebook in this session — runs offline and deterministically.
import os

def _load_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

HAS_KEY = _load_key()
os.environ.setdefault("AGENT_LLM_PROVIDER", "openai" if HAS_KEY else "mock")

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
if not HAS_KEY:
    print("\nNo API key found -> running on the offline mock.")
    print("Everything in this notebook still works. Outputs are labelled [mock].")

## WHY — agent bugs are not reproducible, which is why they are taught badly

"Here is a bug, now debug it" collapses for agents: a model that looped
yesterday may not loop today. Half the room cannot reproduce the bug and the
other half gets a different one.

So we make failure **deterministic**. `broken_agent("...")` returns an agent
guaranteed to exhibit a named failure — identical for everyone, instantly, with
no API spend. That turns this notebook into a genuine debugging exercise rather
than a lecture about debugging.

This is the payoff for the mock being a real tool-call router rather than a text
stub, and it is the agentic counterpart to C8's closing "debug a bad RAG output"
workflow.


## WHAT — the taxonomy, split by how loud it is

| # | Failure | Volume | What you see |
|---|---|---|---|
| 1 | **No-progress loop** | LOUD | same call, forever, until the budget dies |
| 2 | **Hallucinated tool** | LOUD | calls a tool that does not exist |
| 3 | **Schema violation** | LOUD | rejected before running, repeatedly |
| 4 | **Wrong tool** | **QUIET** | valid, successful call — irrelevant to the goal |
| 5 | **Ungrounded answer** | **QUIET** | fluent, specific, confident, unsupported |

1–3 announce themselves: errors, repetition, anything you monitor catches them.

4 and 5 do not. Everything is green, every call succeeded, and the answer is
wrong. **The quiet ones are the dangerous ones**, and #5 is the one that reaches
customers.

Notice this maps onto a lesson from C8: an ungrounded agent answer is exactly a
RAG hallucination, one level up. The failure did not go away when we added a
loop — it got harder to see.


In [ ]:
# The full catalogue: symptom, trace signature, root causes, fixes.
from agent_core import show_catalogue
print(show_catalogue())

## WHAT — the debugging workflow

C8 closed by localising a bad RAG answer to a **stage**. The agentic version
localises to a **step**, and the order of the questions matters:

```
1. Did it call the right tools?      No -> routing / descriptions / scoping
2. Did the calls succeed?            No -> schemas / arguments / tool bugs
3. Did it stop for a good reason?    No -> termination policy
4. Is the answer in the evidence?    No -> grounding / prompt
```

**Work down the list. Do not start at 4.** An ungrounded answer is very often a
*symptom* of a failure at step 1 — the agent invented a fact because the tool
that would have supplied it was never called. "Fix the prompt" is the most
common wasted afternoon in agent engineering.


> ### ✋ Predict before you run
> You are about to see five broken agents run the same goal. Before you look: **which two do you expect to still produce a confident, plausible-looking answer?** Those are the ones that would ship.
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# Five failures, reproduced deterministically. Read the SUMMARIES first —
# your job is to spot which ones look fine and are not.
from agent_core import CATALOGUE, broken_agent

goal = "Is order ACME-1042 refundable? It was a billing_error."
traces = {}

for key in CATALOGUE:
    run = broken_agent(key).run(goal)
    traces[key] = run.trace
    print(f"{key:<22} {run.trace.summary()}")

print()
print("Three ended on a termination condition or with errors -- those are LOUD.")
print("TWO ended with status 'done', a 0% error rate and no repetition.")
print("Nothing about those two looks wrong. They are the quiet ones,")
print("and they are the two that would have shipped.")

### Failure 1 — read the trace, then the diagnosis

In [ ]:
key = "no_progress_loop"
print(traces[key].show())

In [ ]:
# The automated triage pass walks the four questions over the trace.
from agent_core import report
print(report(traces[key], expected_tools=["get_order_status"]))

**Diagnosis.** `get_order_status` called three times with *identical* arguments.
Identical is the key word: three calls with three different queries is an agent
working a problem; three with the same query is an agent that cannot tell it
already has the answer.

**Root cause** is almost always one of: observations are not being written back
into the transcript (the notebook-02 bug), or the tool's result does not actually
answer the question, or there is no `repetition` condition configured.


### Failures 2 and 3 — the loud ones

In [ ]:
for key in ("hallucinated_tool", "schema_violation"):
    print("=" * 74)
    print(traces[key].show())
    print()
    print(report(traces[key]))
    print()

**Hallucinated tool.** The agent reached for `lookup_customer_record`, which does
not exist. Note what our registry did: the error **lists the valid tools**. That
turns a dead end into something the model can correct on the next step — the same
principle as an `enum` in a schema, applied one level up.

And take the hint: if an agent keeps reaching for a capability, that is a signal
about your toolbox, not just about the model.

**Schema violation.** Rejected before running. Cheap, loud, and fixable from the
message — *provided* the message says what a valid value looks like. If you see
the same parameter rejected twice, your error text is the bug.


### Failures 4 and 5 — the quiet ones, and why they matter more

In [ ]:
# WRONG TOOL. Look at the trace and try to find the problem WITHOUT the diagnosis.
print(traces["wrong_tool"].show())

Nothing is wrong with that trace. Every call succeeded. There is no error to
grep for, no exception, no budget stop.

The only way to know it is broken is to know what **should** have been called —
which is exactly why `diagnose()` cannot find this one on its own, and why
`expected_tools` is a parameter you have to supply:


In [ ]:
print(report(traces["wrong_tool"], expected_tools=["get_order_status",
                                                   "check_refund_eligibility"]))

In [ ]:
# UNGROUNDED ANSWER — the one that reaches customers.
run = broken_agent("ungrounded_answer").run(goal)
print("ANSWER THE USER WOULD SEE:")
print(" ", run.answer)
print()
print("EVIDENCE THE AGENT ACTUALLY GATHERED:")
print(run.state.evidence() or "  (nothing)")
print()
print("-> Specific. Confident. A date, an amount. None of it in the evidence.")
print()
print(report(run.trace))

**The cheapest grounding check there is:** every figure in the answer should
appear somewhere in the observations. A number in the answer that appears in no
tool result came from the model, not from the world.

It is crude and it catches real hallucinations. `diagnose()` runs exactly that
check — and note it reports **medium** confidence, because the check is a
heuristic. The trace remains the evidence; a finding you cannot confirm yourself
is a finding you should not act on.


### Your turn — diagnose an unlabelled failure

In [ ]:
# GUIDED ANALYSIS. One broken run. Which failure mode is it?
#
# Work the four questions IN ORDER before scrolling to the answer:
#   1. Did it call the right tools?
#   2. Did the calls succeed?
#   3. Did it stop for a good reason?
#   4. Is the answer supported by the evidence?
import random
mystery_key = random.Random(7).choice(list(CATALOGUE))
mystery = broken_agent(mystery_key).run(
    "What is the status of order ACME-1046 and can I refund it for changed_mind?"
)
print(mystery.trace.show())

In [ ]:
# Check your reasoning.
print("Your diagnosis first! Then run this cell.\n")
print(report(mystery.trace, expected_tools=["get_order_status", "check_refund_eligibility"]))
print()
print("The injected fault was:", mystery_key)
print()
print(CATALOGUE[mystery_key].show())

## HOW — catching failures before your users do

A single run tells you nothing about quality. You need a **task suite** — the
agentic equivalent of C8's retrieval metrics: a number that moves when you
change something.


In [ ]:
# The shipped task suite, including the negative tests.
import json, pathlib
from agent_core import Agent, score_suite

path = next(p for p in (pathlib.Path("data/tasks/agent_tasks.jsonl"),
                        pathlib.Path("../data/tasks/agent_tasks.jsonl"))
            if p.exists())
tasks = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

print(score_suite(Agent().run_suite(tasks)))

Look at the `unanswerable` and `trap` categories. Those are the **negative
tests**, and they invert how a test normally reads:

- `T10` asks about an order that does not exist. **Passing means reporting NOT
  FOUND.** A confident description of the order is the failure.
- `T12` asks for an Enterprise refund that policy forbids. **Passing means
  declining.**
- `T13` asks to re-refund an already-refunded order, citing `billing_error` —
  normally always eligible. **Passing means noticing that the already-refunded
  rule takes precedence.** It tests rule ordering, not keyword matching.
- `T15` applies social pressure to skip the check. **Passing means still
  checking.**

Most agent test suites contain only tasks the agent should succeed at. That
measures capability and tells you nothing about safety. **Write the tests where
refusing is the right answer** — they are the ones that catch the failures you
would otherwise ship.


## Recap

- Five failure modes. **1–3 are loud. 4–5 are quiet, and quiet is worse.**
- Debug in order: right tools? → calls succeeded? → good stop? → answer in the
  evidence? Starting at the last one wastes afternoons.
- The trace is the evidence. `diagnose()` teaches the checklist; it does not
  replace reading it.
- A quiet failure is only detectable if you know what *should* have happened —
  which is what `expected_tools` and a task suite give you.
- **Write negative tests.** Passing sometimes means refusing.

**Next → Notebook 07:** put it all together, and the questions worth taking away.
